In [1]:
import pandas as pd
import os

# ============================
# Pfade
# ============================
input_path = r"K:\Team\Böhmer_Michael\PAPER\Philipp\Datensatz\Basistabelle_5_bis_14_PostOP_outlier.xlsx"
output_dir = r"K:\Team\Böhmer_Michael\PAPER\Philipp\Datensatz\ML_Datasets"

target_col = "Verletzungsstatus"

# ============================
# Feature-Listen
# ============================
top38_features = [
    "CMJ_KAI ecc",
    "CMJ_KAI con",
    "INV_ISO_Arbeit_Unterschied Extension Flexion",
    "UNINV_ISO_Max Flexion",
    "INV_ISO_Drehmoment_Unterschied Extension Flexion",
    "UNINV_CMJ_uni_Relative Peak Landing Force-Mittelwert [BW]",
    "INV_CMJ_uni_Max Rate of Force Development-Mittelwert [N/s]",
    "INV_ISO_Drehmoment_Verhaeltnis Flexion Extension",
    "INV_Arbeit_Flexion",
    "UNINV_CMJ_uni_Bremsimpuls-Mittelwert [N*s]",
    "INV_ISO_Arbeit_Verhaeltnis Flexion Extension",
    "UNINV_CMJ_uni_Av. propulsive force",
    "INV_CMJ_uni_Peak braking force",
    "ISO_Arbeit_Seitenunterschied Flexion absolut",
    "CMJ_Braking duration",
    "CMJ_Countermovement depth",
    "UNINV_CMJ_uni_Relative Peak Loading Force-Mittelwert [BW]",
    "CMJ_Jump Height impulse",
    "CMJ_Rel. peak loading force",
    "CMJ_Propulsive duration",
    "INV_Arbeit_Extension",
    "INV_CMJ_uni_Net Impulse-Mittelwert [N*s]",
    "CMJ_Vertical Stiffness",
    "UNINV_CMJ_uni_Braking Duration-Mittelwert [s]",
    "UNINV_ISO_Drehmoment_Verhaeltnis Flexion Extension",
    "ISO_Drehmoment_Seitenunterschied Extension absolut",
    "UNINV_CMJ_uni_Max Rate of Force Development-Mittelwert [N/s]",
    "CMJ_Jump Height flighttime",
    "CMJ_Rel. Peak landing force",
    "INV_CMJ_uni_Av. propulsive force",
    "INV_CMJ_uni_Durchschnittliche Bremsleistung-Mittelwert [Watt]",
    "ISO_Drehmoment_Seitenunterschied Extension relativ",
    "UNINV_CMJ_uni_Peak Braking Force-Mittelwert [N]",
    "INV_CMJ_uni_Relative Peak Loading Force-Mittelwert [BW]",
    "INV_CMJ_uni_Durchschnittliche Bremsgeschwindigkeit-Mittelwert [m/s]",
    "UNINV_ISO_Drehmoment_Unterschied Extension Flexion",
    "Geschlecht_weiblich",
    "INV_CMJ_uni_Vertical Stiffness-Mittelwert [kN/m]",
]

stable70_features = [
    "Geschlecht_weiblich",
    "CMJ_KAI ecc",
    "CMJ_KAI con",
    "INV_ISO_Arbeit_Unterschied Extension Flexion",
    "UNINV_ISO_Max Flexion",
    "INV_ISO_Drehmoment_Unterschied Extension Flexion",
    "UNINV_CMJ_uni_Relative Peak Landing Force-Mittelwert [BW]",
    "INV_CMJ_uni_Max Rate of Force Development-Mittelwert [N/s]",
    "INV_ISO_Drehmoment_Verhaeltnis Flexion Extension",
    "INV_Arbeit_Flexion",
    "UNINV_CMJ_uni_Bremsimpuls-Mittelwert [N*s]",
    "INV_ISO_Arbeit_Verhaeltnis Flexion Extension",
    "UNINV_CMJ_uni_Av. propulsive force",
    "INV_CMJ_uni_Peak braking force",
    "ISO_Arbeit_Seitenunterschied Flexion absolut",
]

# ============================
# Daten laden
# ============================
df = pd.read_excel(input_path)

# Spaltennamen säubern
df.columns = df.columns.str.strip()

# Feature-Listen säubern
top38_features = [f.strip() for f in top38_features]
stable70_features = [f.strip() for f in stable70_features]

# ============================
# Checks
# ============================
if target_col not in df.columns:
    raise ValueError(f"Zielvariable '{target_col}' nicht gefunden.")

missing_top38 = [c for c in top38_features if c not in df.columns]
missing_stable70 = [c for c in stable70_features if c not in df.columns]

print("Missing Top38:", missing_top38)
print("Missing Stable70:", missing_stable70)

if missing_top38 or missing_stable70:
    raise ValueError("Feature-Namen stimmen nicht mit den Spalten überein.")

# ============================
# Datensätze erstellen
# ============================
df_top38 = df[[target_col] + top38_features].copy()
df_stable70 = df[[target_col] + stable70_features].copy()

# ============================
# Speichern
# ============================
os.makedirs(output_dir, exist_ok=True)

top38_path = os.path.join(output_dir, "dataset_top38.xlsx")
stable70_path = os.path.join(output_dir, "dataset_stable70.xlsx")

df_top38.to_excel(top38_path, index=False)
df_stable70.to_excel(stable70_path, index=False)

print("\n✔ Datensätze erfolgreich erstellt:")
print("Top-38:", top38_path)
print("Stable-70:", stable70_path)

Missing Top38: []
Missing Stable70: []

✔ Datensätze erfolgreich erstellt:
Top-38: K:\Team\Böhmer_Michael\PAPER\Philipp\Datensatz\ML_Datasets\dataset_top38.xlsx
Stable-70: K:\Team\Böhmer_Michael\PAPER\Philipp\Datensatz\ML_Datasets\dataset_stable70.xlsx


### SHAP Setup

In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import shap

def collect_shap_data_lr(
    data_path,
    target_column="Verletzungsstatus",
    cv_splits=5,
    cv_repeats=10,
    shap_background_size=100,
    verbose=False
):
    """
    Trainiert LR mit CV, sammelt SHAP-Werte (log-odds) und gibt zurück:
      - features: Feature-Namen in Originalreihenfolge
      - shap_stack: gestackte SHAP-Matrix (alle Folds, alle Test-Samples)
      - X_stack:   gestackte (skalierte) Test-Features
      - exp_val:   expected_value (Skalar) für die positive Klasse (1)
      - last_model, last_test_len: wie zuvor
      - signed_mean_shaps: mean(SHAP) je Feature (Richtung)
      - abs_mean_shaps:    mean(|SHAP|) je Feature (robuste Wichtigkeit)
    """
    # 1) Daten einlesen
    df = pd.read_excel(data_path)
    y  = df[target_column].astype(int)
    X  = df.drop(columns=[target_column])
    features = X.columns.tolist()

    # 2) CV-Setup
    cv = RepeatedStratifiedKFold(n_splits=cv_splits, n_repeats=cv_repeats, random_state=42)
    all_shap = []
    all_X    = []
    last_exp_val = None

    if verbose:
        print("→ Starte LR + SHAP (log-odds)…")

    rng_global = np.random.default_rng(42)

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        if verbose: print(f"Fold {fold}")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # 3) Skalierung (kein Leakage), aber Dummy-Variable nicht skalieren
        dummy_col = "Geschlecht_weiblich"

        cols_to_scale = [c for c in X_train.columns if c != dummy_col]

        X_train_s = X_train.copy()
        X_test_s = X_test.copy()

        scaler = StandardScaler()
        X_train_s[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
        X_test_s[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

        # Für sklearn/shap als Array weitergeben
        X_train_s = X_train_s.values
        X_test_s = X_test_s.values

        # 4) Logistische Regression
        lr = LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=1000,
            C=0.842,
            class_weight=None,
            random_state=42
        )
        lr.fit(X_train_s, y_train)

        last_model   = lr
        last_test_len = X_test_s.shape[0]

        # 5) SHAP: LinearExplainer (log-odds), Background aus TRAIN
        n_bg = min(shap_background_size, len(X_train_s))
        bg_idx = rng_global.choice(len(X_train_s), size=n_bg, replace=False)
        bg = X_train_s[bg_idx, :]

        expl = shap.LinearExplainer(
            lr, bg,
            feature_names=features,
            model_output="log_odds"
        )
        sv = expl(X_test_s)

        vals = np.nan_to_num(sv.values, nan=0.0, posinf=0.0, neginf=0.0)
        all_shap.append(vals)
        all_X.append(X_test_s)

        # expected_value robust für Klasse 1
        exp_raw = expl.expected_value
        if isinstance(exp_raw, (list, tuple, np.ndarray)):
            pos_idx = int(np.where(lr.classes_ == 1)[0][0])
            last_exp_val = float(np.ravel(exp_raw)[pos_idx])
        else:
            last_exp_val = float(exp_raw)

    # 6) Stacking
    shap_stack = np.vstack(all_shap)  # shape: [N_total, n_features]
    X_stack    = np.vstack(all_X)

    # 7) Kennzahlen für Ranking & Richtung:
    signed_mean_shaps = np.nanmean(shap_stack, axis=0)          # mean(SHAP)
    abs_mean_shaps    = np.nanmean(np.abs(shap_stack), axis=0)  # mean(|SHAP|)

    return (
        features,
        shap_stack,
        X_stack,
        last_exp_val,
        last_model,
        last_test_len,
        signed_mean_shaps,
        abs_mean_shaps
    )

features, shap_stack, X_stack, exp_val, last_model, last_test_len, signed_mean_shaps, abs_mean_shaps = collect_shap_data_lr(
    data_path            = r"K:\Team\Böhmer_Michael\PAPER\Datensätze_final\korrigiert\ML_Maestroni_korrigiert.xlsx",
    target_column        = "Injury status",
    cv_splits            = 5,
    cv_repeats           = 10,
    shap_background_size = 100,
    use_class_weight     = False,
    max_iter             = 200,
    C                    = 1.0,
    verbose              = False
)

print("\nSHAP-Wertprüfung:")
print("→ Min:", shap_stack.min(), "Max:", shap_stack.max(), "Mean:", shap_stack.mean())
print("→ Anzahl NaN:", np.isnan(shap_stack).sum(), "Inf:", np.isinf(shap_stack).sum())

# Ranking nach mean(|SHAP|)
order = np.argsort(abs_mean_shaps)[::-1]
print("\nAlle Features (sortiert nach mean(|SHAP|)):")

header = f"{'Feature':<60} {'signed_mean':>15} {'mean(|SHAP|)':>15}"
print(header)
print("-" * len(header))

for i in order:   # statt order[:20]
    print(f"{features[i]:<60} {signed_mean_shaps[i]:+15.6f} {abs_mean_shaps[i]:15.6f}")



### SHAP Test

In [ ]:
from scipy.special import expit
import numpy as np

# Indexbereich des letzten Folds im gestackten Array
start = len(X_stack) - last_test_len
end   = len(X_stack)

rng = np.random.default_rng(42)
idx_local = rng.choice(np.arange(start, end), size=min(5, last_test_len), replace=False)

p_from_shap   = []
p_from_model  = []

for i in idx_local:
    logit_i = exp_val + shap_stack[i].sum()        # log-odds rekonstruiert
    p_from_shap.append(expit(logit_i))             # -> Wahrscheinlichkeit
    p_from_model.append(last_model.predict_proba(X_stack[i].reshape(1, -1))[0, 1])

p_from_shap  = np.array(p_from_shap)
p_from_model = np.array(p_from_model)
diff = p_from_shap - p_from_model
print("\n=== SHAP Selbsttest (letzter Fold, Zufalls-Samples) ===")
print("Abweichung = P(rekonstruiert aus SHAP) – P(Modell)\n")
print(f"Maximale Abweichung : {np.max(np.abs(diff)):.3e}")
print(f"Mittlere Abweichung : {np.mean(np.abs(diff)):.3e}")
print(f"Median Abweichung   : {np.median(np.abs(diff)):.3e}\n")

for i in range(len(idx_local)):
    print(f"P_SHAP={p_from_shap[i]:.6f} | "
          f"P_Modell={p_from_model[i]:.6f} | "
          f"Δ={diff[i]:+.2e}")



### Globale SHAP Log odds

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import shap

# -----------------------------
# Parameter: Ranking-Auswahl
# -----------------------------
rank_by = "abs"   # "abs" oder "signed"
wrap_width = 40
row_height = 0.35
w = 11.0

# -----------------------------
# Reihenfolge basierend auf rank_by
# -----------------------------
if rank_by not in ("abs", "signed"):
    raise ValueError("rank_by must be 'abs' or 'signed'.")

if rank_by == "abs":
    ranking_scores = abs_mean_shaps.copy()                 # mean(|SHAP|)
else:  # rank_by == "signed"
    ranking_scores = np.abs(signed_mean_shaps).copy()      # |mean(SHAP)| für Sortierung

order_bar = np.argsort(ranking_scores)[::-1]               # absteigend                             
order_bee = order_bar                                      # Beeswarm: Top oben

# Einheitliche Höhe pro Feature
h = row_height * len(order_bee) + 2.0

# Einheitliche, weich umgebrochene Labels in identischer Reihenfolge
wrapped_names = [textwrap.fill(features[i], width=wrap_width) for i in order_bee]

# =============================
# 1) BEESWARM
# =============================
shap.summary_plot(
    shap_stack[:, order_bee],
    features=X_stack[:, order_bee],
    feature_names=wrapped_names,
    plot_type="dot",
    max_display=len(order_bee),
    sort=False,
    color_bar=True,
    plot_size=(w, h),
    show=False
)

ax = plt.gca()
ax.tick_params(axis='y', labelsize=11, pad=10)
ax.tick_params(axis='x', labelsize=11)
plt.gcf().subplots_adjust(left=0.35, right=0.96, top=0.96, bottom=0.06)
plt.tight_layout()
plt.show()

# =============================
# 2) BALKENPLOT (zweite Grafik)
# =============================

w2 = 14.0        # Breite bei Bedarf weiter erhöhen
pad_frac = 0.22  # zusätzlicher Platz rechts/links für Labels

fig, ax = plt.subplots(figsize=(w2, h))

y = np.arange(len(order_bee))

if rank_by == "signed":
    # Anzeige: signed_mean (mit Vorzeichen), zentriert um 0
    vals = signed_mean_shaps[order_bee]
    max_abs = np.nanmax(np.abs(vals)) if vals.size else 1.0
    ax.barh(y, vals)

    # Werte-Label am Balkenende
    eps = 0.02 * max_abs
    for yi, v in zip(y, vals):
        if np.isfinite(v) and v != 0:
            ax.text(v + (eps if v > 0 else -eps), yi, f"{v:+.3f}",
                    va='center', ha=('left' if v > 0 else 'right'), fontsize=9)
        else:
            ax.text(0 + eps, yi, f"{0:+.3f}", va='center', ha='left', fontsize=9)

    # symmetrische Achse mit Extra-Puffer
    ax.set_xlim(-max_abs * (1 + pad_frac), max_abs * (1 + pad_frac))
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel("Mean SHAP (log-odds, signed)")

else:
    # Anzeige: mean(|SHAP|)
    vals = abs_mean_shaps[order_bee]
    vmax = np.nanmax(vals) if vals.size else 1.0
    ax.barh(y, vals)

    eps = 0.02 * vmax
    for yi, v in zip(y, vals):
        v = 0.0 if not np.isfinite(v) else v
        ax.text(v + eps, yi, f"{v:.3f}", va='center', ha='left', fontsize=9)

    ax.set_xlim(0, vmax * (1 + pad_frac))
    ax.set_xlabel("Mean |SHAP| (log-odds)")

# Y-Achse und Layout wie im Beeswarm
ax.set_yticks(y)
ax.set_yticklabels(wrapped_names)
ax.invert_yaxis()
ax.tick_params(axis='y', labelsize=11, pad=10)
ax.tick_params(axis='x', labelsize=11)
fig.subplots_adjust(left=0.35, right=0.985, top=0.96, bottom=0.06)
plt.tight_layout()
plt.show()


